# Clean Artifact-Level Table

This notebook prepares the endpoint-security data for later Neo4j loading.

The goal here is **only data cleaning and artifact-level aggregation**. No Neo4j code is included.

### Artifact identity

An artifact is identified by its **SHA-256 hash**. Multiple filenames can therefore belong to the same artifact.

The current analysis scope is the 22 endpoints used in the existing artifact-level table, and only records explicitly marked `FileType: Malicious` are included.

In [13]:
from pathlib import Path
import pandas as pd
from IPython.display import display

SOURCE_PATH = Path("../data/endpoint_security.csv")
OUTPUT_PATH = Path("../data/artifact_level_table.csv")

TARGET_ENDPOINTS = [
    "EPPGWIOF", "EPJ76RZV", "EP4YIFHV", "EPAGTKNF", "EPAJI3E5",
    "EP7UYCXM", "EPEEHWSG", "EPGSQVEB", "EPJ42VCZ", "EPMDU6L5",
    "EPMVZYZO", "EPNPX6VY", "EPQBJBDX", "EPQKOBYG", "EPRD7FJT",
    "EPSES7ZK", "EPSJWX5J", "EPT3KTUL", "EPVC2P55", "EPVSWBNA",
    "EPWZ7L6L", "EPZRAPEL"
]

In [14]:
# Load the source dataset.
df = pd.read_csv(SOURCE_PATH)

print('Raw rows:', len(df))
print('Columns:', df.columns.tolist())
display(df.head())


Raw rows: 23540
Columns: ['endpoint_id', 'endpoint_name', 'mac', 'unit', 'log_type', 'event_time', 'received_time', 'message']


,endpoint_id,endpoint_name,mac,unit,log_type,event_time,received_time,message
0,EPZRAPEL,host-d14b,8A:58:3E:18:FD:1B,unit-foxtrot,quarantine,01-09-2026 04:58:58,01-09-2026 04:58:56,No suspicious file found by type or extension.
1,EPZRAPEL,host-d14b,8A:58:3E:18:FD:1B,unit-foxtrot,quarantine,01-09-2026 04:58:58,01-09-2026 04:58:56,File: 53EDA42869D426262B752F088EA605187284ACE8...
2,EPYJFFRM,host-2864,F2:EA:9C:78:DC:3C,unit-oscar,quarantine,01-09-2026 05:20:37,01-09-2026 05:20:39,No suspicious file found by type or extension.
3,EPYJFFRM,host-2864,F2:EA:9C:78:DC:3C,unit-oscar,quarantine,01-09-2026 05:20:37,01-09-2026 05:20:39,No Malicious files found.
4,EPFFTO5T,host-508b,FA:2F:46:B7:14:23,unit-delta,quarantine,01-09-2026 08:50:38,01-09-2026 08:50:38,No suspicious file found by type or extension.


## 1. Select the artifact observations

The source message contains the file name and SHA-256. We keep only:

- endpoints in the defined analysis scope
- messages containing `FileType: Malicious`
- valid 64-character SHA-256 values


In [15]:
scope = df[
    df['endpoint_id'].isin(TARGET_ENDPOINTS)
    & df['message'].fillna('').str.contains('FileType: Malicious', case=False, na=False)
].copy()

scope['sha256'] = scope['message'].str.extract(
    r'SHA256:\s*([0-9a-fA-F]{64})', expand=False
).str.lower()

scope['file_name'] = scope['message'].str.extract(
    r'^File:\s*(.*?)\s*\|', expand=False
)

scope['event_time'] = pd.to_datetime(
    scope['event_time'],
    format='%d-%m-%Y %H:%M:%S',
    errors='coerce'
)

scope = scope.dropna(subset=['sha256', 'file_name']).copy()

print('Selected observation rows:', len(scope))
print('Unique endpoints:', scope['endpoint_id'].nunique())
print('Unique SHA-256 artifacts:', scope['sha256'].nunique())


Selected observation rows: 87
Unique endpoints: 22
Unique SHA-256 artifacts: 49


In [16]:
# Endpoint + filename observation table.
endpoint_file_level = (
    scope.groupby(['endpoint_id', 'sha256', 'file_name'], as_index=False)
    .agg(
        record_count=('sha256', 'size'),
        first_event=('event_time', 'min'),
        last_event=('event_time', 'max'),
    )
    .sort_values(['endpoint_id', 'sha256', 'file_name'])
    .reset_index(drop=True)
)

print('Endpoint-file rows:', len(endpoint_file_level))
display(endpoint_file_level.head(20))


Endpoint-file rows: 53


,endpoint_id,sha256,file_name,record_count,first_event,last_event
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,load-hub-i18n.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,shopping_iframe_driver.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,tokenized-card.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,wallet_checkout_autofill_driver.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,wallet-buynow.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
5,EP4YIFHV,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,wallet_donation_driver.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
6,EP4YIFHV,569e425e7f048b1f769c22d65aa91427b69c3d248068af...,runtime.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
7,EP4YIFHV,56bada50510d97878d514009279c3a662d9894a21c344f...,load-ec-i18n.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
8,EP4YIFHV,620389d26167bea08e56a90480f74d3dbd32c3e974f4de...,bnpl.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09
9,EP4YIFHV,6c21396c9518fa414ddb4dada2adb09e2d2b2fc5069d89...,bnpl_driver.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09


## 2. Build the artifact-level table

One row = one unique SHA-256 artifact.

Important: filenames are aggregated rather than selecting the first filename. This preserves cases where the same SHA-256 appears under multiple filenames.

In [17]:
artifact_level = (
    scope.groupby('sha256', as_index=False)
    .agg(
        file_names=('file_name', lambda x: sorted(set(x.dropna()))),
        record_count=('sha256', 'size'),
        first_event=('event_time', 'min'),
        last_event=('event_time', 'max'),
        endpoint_count=('endpoint_id', 'nunique'),
        endpoint_ids=('endpoint_id', lambda x: sorted(set(x))),
    )
    .sort_values('sha256')
    .reset_index(drop=True)
)

artifact_level['file_name_count'] = artifact_level['file_names'].apply(len)

artifact_level = artifact_level[
    ['sha256', 'file_names', 'file_name_count', 'record_count',
     'first_event', 'last_event', 'endpoint_count', 'endpoint_ids']
]

print('Artifact-level rows:', len(artifact_level))
display(artifact_level)


Artifact-level rows: 49


,sha256,file_names,file_name_count,record_count,first_event,last_event,endpoint_count,endpoint_ids
0,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],1,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV]
1,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],1,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV]
2,18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"[Buy CP PLUS Dashboard Camera, Warranty NONE 1...",1,2,2026-09-03 19:03:33,2026-09-04 18:10:01,1,[EPAGTKNF]
3,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...,[53EDA42869D426262B752F088EA605187284ACE8],1,1,2026-09-01 04:58:58,2026-09-01 04:58:58,1,[EPZRAPEL]
4,30c78bc9ec28a6e4d47708c309f8c88599c71a0fb379a3...,[dash cam.pdf],1,1,2026-09-24 08:48:40,2026-09-24 08:48:40,1,[EPQBJBDX]
5,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],1,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV]
6,404ee57531c33156d62154f49e3700c8c1f09cd0c75692...,[Final_Documents.zip],1,1,2026-09-11 10:11:45,2026-09-11 10:11:45,1,[EPGSQVEB]
7,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],1,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV]
8,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],1,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV]
9,528c866fabbe17c30e9790a12ac09b94dd820ba4e46b0f...,[wallet_donation_driver.js],1,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1,[EP4YIFHV]


# Endpoint + filename observation table.
endpoint_file_level = (
    scope.groupby(['endpoint_id', 'sha256', 'file_name'], as_index=False)
    .agg(
        record_count=('sha256', 'size'),
        first_event=('event_time', 'min'),
        last_event=('event_time', 'max'),
    )
    .sort_values(['endpoint_id', 'sha256', 'file_name'])
    .reset_index(drop=True)
)

print('Endpoint-file rows:', len(endpoint_file_level))
display(endpoint_file_level.head(20))


In [18]:
assert scope['endpoint_id'].nunique() == 22
assert len(scope) == 87
assert artifact_level['sha256'].nunique() == 49
assert artifact_level['record_count'].sum() == 87
assert endpoint_file_level['record_count'].sum() == 87
assert (artifact_level['file_name_count'] >= 1).all()

print('Validation passed.')
print('  Observation rows :', len(scope))
print('  Endpoints        :', scope['endpoint_id'].nunique())
print('  Unique artifacts :', artifact_level['sha256'].nunique())
print('  Total records    :', artifact_level['record_count'].sum())
            
print('\nRecord-count distribution:')
print(artifact_level['record_count'].value_counts().sort_index())


Validation passed.
  Observation rows : 87
  Endpoints        : 22
  Unique artifacts : 49
  Total records    : 87

Record-count distribution:
record_count
1    22
2    23
4     1
5     3
Name: count, dtype: int64


assert scope['endpoint_id'].nunique() == 22
assert len(scope) == 87
assert artifact_level['sha256'].nunique() == 49
assert artifact_level['record_count'].sum() == 87
assert endpoint_file_level['record_count'].sum() == 87
assert (artifact_level['file_name_count'] >= 1).all()

print('Validation passed.')
print('  Observation rows :', len(scope))
print('  Endpoints        :', scope['endpoint_id'].nunique())
print('  Unique artifacts :', artifact_level['sha256'].nunique())
print('  Total records    :', artifact_level['record_count'].sum())
print('\nRecord-count distribution:')
print(artifact_level['record_count'].value_counts().sort_index())


In [19]:
artifact_level_csv = artifact_level.copy()
artifact_level_csv['file_names'] = artifact_level_csv['file_names'].apply(lambda x: ' | '.join(x))
artifact_level_csv['endpoint_ids'] = artifact_level_csv['endpoint_ids'].apply(lambda x: ' | '.join(x))

artifact_level_csv.to_csv(OUTPUT_PATH, index=False)
            
print(f'Saved: {OUTPUT_PATH.resolve()}')


Saved: C:\Users\Asus\Desktop\web_activity\data\artifact_level_table.csv


artifact_level_csv = artifact_level.copy()
artifact_level_csv['file_names'] = artifact_level_csv['file_names'].apply(lambda x: ' | '.join(x))
artifact_level_csv['endpoint_ids'] = artifact_level_csv['endpoint_ids'].apply(lambda x: ' | '.join(x))

artifact_level_csv.to_csv(OUTPUT_PATH, index=False)
print(f'Saved: {OUTPUT_PATH.resolve()}')


In [20]:
print(scope.columns.tolist())
print(scope.shape)
display(scope.head())

['endpoint_id', 'endpoint_name', 'mac', 'unit', 'log_type', 'event_time', 'received_time', 'message', 'sha256', 'file_name']
(87, 10)


,endpoint_id,endpoint_name,mac,unit,log_type,event_time,received_time,message,sha256,file_name
1,EPZRAPEL,host-d14b,8A:58:3E:18:FD:1B,unit-foxtrot,quarantine,2026-09-01 04:58:58,01-09-2026 04:58:56,File: 53EDA42869D426262B752F088EA605187284ACE8...,2e4dff1672bbabba74a0a99d54e38e56e036ce84719895...,53EDA42869D426262B752F088EA605187284ACE8
2997,EPAGTKNF,host-227c,B2:F1:48:20:04:E9,unit-quebec,quarantine,2026-09-04 18:10:01,04-09-2026 18:09:57,"File: Buy CP PLUS Dashboard Camera, Warranty N...",18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"Buy CP PLUS Dashboard Camera, Warranty NONE 1 ..."
3415,EPAGTKNF,host-227c,B2:F1:48:20:04:E9,unit-quebec,quarantine,2026-09-03 19:03:33,03-09-2026 19:03:31,"File: Buy CP PLUS Dashboard Camera, Warranty N...",18866b6f7308f73bc1b63193e50629bfd4fff701d6b9d3...,"Buy CP PLUS Dashboard Camera, Warranty NONE 1 ..."
4998,EPJ42VCZ,host-56a6,FE:1B:B6:08:36:73,unit-india,quarantine,2026-09-07 08:43:44,07-09-2026 08:43:49,File: RAGHAV STNC STNA.pdf | Quarantined At: 2...,a2beca1870ae689f4b046a20c3d8e2e48557d782dea534...,RAGHAV STNC STNA.pdf
4999,EPJ42VCZ,host-56a6,FE:1B:B6:08:36:73,unit-india,quarantine,2026-09-07 08:43:44,07-09-2026 08:43:49,File: RAGHAV STNA STNB.pdf | Quarantined At: 2...,854b8f7716627df333e8ecfeae5887c02b3195658f7394...,RAGHAV STNA STNB.pdf


In [21]:
endpoint_artifact_edges = (
    scope
    .groupby(["endpoint_id", "sha256"], as_index=False)
    .agg(
        file_names=("file_name", lambda x: sorted(set(x.dropna()))),
        record_count=("sha256", "size"),
        first_event=("event_time", "min"),
        last_event=("event_time", "max"),
    )
)

endpoint_artifact_edges["file_name_count"] = (
    endpoint_artifact_edges["file_names"].apply(len)
)

display(endpoint_artifact_edges.head())

,endpoint_id,sha256,file_names,record_count,first_event,last_event,file_name_count
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,[load-hub-i18n.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,[shopping_iframe_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,[tokenized-card.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,[wallet_checkout_autofill_driver.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,[wallet-buynow.bundle.js],2,2026-09-11 09:23:33,2026-09-11 09:35:09,1


In [22]:
print("Endpoint-artifact edges:", len(endpoint_artifact_edges))
print("Endpoints:", endpoint_artifact_edges["endpoint_id"].nunique())
print("Artifacts:", endpoint_artifact_edges["sha256"].nunique())
print("Total records:", endpoint_artifact_edges["record_count"].sum())

Endpoint-artifact edges: 49
Endpoints: 22
Artifacts: 49
Total records: 87


In [23]:
EDGE_OUTPUT_PATH = Path("../data/endpoint_artifact_edges.csv")

edge_export = endpoint_artifact_edges.copy()

edge_export["file_names"] = (
    edge_export["file_names"]
    .apply(lambda x: " | ".join(x))
)

edge_export.to_csv(EDGE_OUTPUT_PATH, index=False)

print(f"Saved: {EDGE_OUTPUT_PATH.resolve()}")

Saved: C:\Users\Asus\Desktop\web_activity\data\endpoint_artifact_edges.csv


In [24]:
edge_check = pd.read_csv(EDGE_OUTPUT_PATH)

print("Rows:", len(edge_check))
print("Columns:", edge_check.columns.tolist())

display(edge_check.head())

Rows: 49
Columns: ['endpoint_id', 'sha256', 'file_names', 'record_count', 'first_event', 'last_event', 'file_name_count']


,endpoint_id,sha256,file_names,record_count,first_event,last_event,file_name_count
0,EP4YIFHV,036965c24ac54284e0bfac480395c91214e7b3c64d0021...,load-hub-i18n.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
1,EP4YIFHV,100d970fc784e9bcb37b707ec4ef69825b165a073659f4...,shopping_iframe_driver.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
2,EP4YIFHV,39f5d83d7982cfae057053b0dcffa66fca12bee9769b15...,tokenized-card.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
3,EP4YIFHV,444d4db051edabd5abb29d93dcc5b07e0c662b03e2d785...,wallet_checkout_autofill_driver.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
4,EP4YIFHV,4aab5c4ac7a5cf0aab311362514cda20a2f3beb777a553...,wallet-buynow.bundle.js,2,2026-09-11 09:23:33,2026-09-11 09:35:09,1
